# 11_run_formal_mixed_experiment_fixed_v2

这是一份重新整理后的**正式实验 notebook**，不再依赖容易报错的 `src/pipeline/window_experiment.py` 主流程，
而是直接在 notebook 内实现一套**更稳健的批量窗口实验流程**。

这版重点解决了之前出现的这些问题：

1. `window_binary_data/window_xxx_binary.csv` 不存在  
2. 窗口二值数据长度和 `transactions` / `labels` 对不齐  
3. 稀疏窗口下 `attack_rules=0` 或 `normal_rules=0` 导致规则池直接报错  
4. `RUN_MODE`、`window_ids`、`plan_df` 未定义导致 notebook 某格单独运行就报错  
5. 某一个窗口失败就让整批实验中断  
6. 失败窗口缺少清晰的错误日志和成功窗口列表  

这份 notebook 默认支持三种模式：

- `smoke`：low / mid / high 各取一个窗口
- `custom_batch`：手工指定一小批窗口
- `full`：跑计划表中的全部窗口


In [ ]:

# ===== 0. 基础导入 =====
from pathlib import Path
import sys
import traceback
import pickle
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.data.transaction_utils import build_transactions
from src.rl.state_utils import build_initial_rule_state
from src.rl.ac_model import ActorCriticNet
from src.rl.simple_env import SimpleRuleEnv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LABEL_ATTACK = "LABEL_ATTACK"
LABEL_NORMAL = "LABEL_NORMAL"

print("ROOT =", ROOT)
print("device =", device)


In [ ]:

# ===== 1. 自动寻找窗口计划表 =====
candidate_paths = [
    ROOT / "outputs" / "logs" / "formal_mixed_windows_smoke_test.csv",
    ROOT / "outputs" / "logs" / "formal_selected_mixed_windows.csv",
    ROOT / "data" / "stream" / "swat" / "formal_mixed_windows_plan.csv",
]

plan_path = None
for p in candidate_paths:
    if p.exists():
        plan_path = p
        break

if plan_path is None:
    raise FileNotFoundError(
        "没有找到可用的窗口计划表，请检查这几个文件是否存在：\n"
        + "\n".join(str(p) for p in candidate_paths)
    )

plan_df = pd.read_csv(plan_path)
assert "window_id" in plan_df.columns, f"计划表缺少 window_id 列: {plan_path}"

print("using plan_path =", plan_path)
print("num rows =", len(plan_df))
display(plan_df.head())


In [ ]:

# ===== 2. 选择运行模式 =====
RUN_MODE = "full"   # 可选: "smoke" / "custom_batch" / "full"

CUSTOM_WINDOW_IDS = [1293, 1353, 1124, 1434, 1065, 1002, 1219, 1254]

if RUN_MODE == "smoke":
    if "attack_bin" in plan_df.columns:
        chosen = []
        for b in ["low", "mid", "high"]:
            part = plan_df[plan_df["attack_bin"] == b].copy()
            if len(part) > 0:
                chosen.append(int(part.iloc[0]["window_id"]))
        window_ids = chosen
    else:
        window_ids = plan_df["window_id"].astype(int).tolist()[:3]

elif RUN_MODE == "custom_batch":
    window_ids = [int(x) for x in CUSTOM_WINDOW_IDS]

elif RUN_MODE == "full":
    window_ids = plan_df["window_id"].astype(int).tolist()

else:
    raise ValueError(f"未知 RUN_MODE: {RUN_MODE}")

print("RUN_MODE =", RUN_MODE)
print("window_ids =", window_ids)
print("num windows =", len(window_ids))

check_df = plan_df[plan_df["window_id"].isin(window_ids)].copy()
print("matched in plan_df =", len(check_df))
if len(check_df) > 0:
    cols = [c for c in ["window_id", "n_attack", "n_normal", "attack_ratio", "attack_bin"] if c in check_df.columns]
    display(check_df[cols].sort_values("attack_ratio") if "attack_ratio" in check_df.columns else check_df)


In [ ]:

# ===== 3. 配置实验参数 =====
@dataclass
class RuleMiningConfig:
    lag: int = 2
    min_support: float = 0.1
    min_confidence: float = 0.6
    top_attack_rules: int = 10
    top_normal_rules: int = 10
    relaxed_normal_rules: int = 3
    attack_weight_cap: float = 3.0
    relaxed_normal_min_weight: float = 0.3
    relaxed_normal_weight_scale: float = 30.0
    relaxed_normal_weight_cap: float = 3.0

@dataclass
class WarmStartConfig:
    epochs: int = 200
    lr: float = 1e-3
    seed: int = 42

@dataclass
class EvalConfig:
    random_trials: int = 20
    seed: int = 42

mining_config = RuleMiningConfig()
warmstart_config = WarmStartConfig()
eval_config = EvalConfig()

np.random.seed(warmstart_config.seed)
torch.manual_seed(warmstart_config.seed)

print(mining_config)
print(warmstart_config)
print(eval_config)


In [ ]:

# ===== 4. 自动补齐 / 重建 window_binary_data（带 lag 缓冲）=====
X_binary = pd.read_csv(ROOT / "data" / "processed" / "swat" / "X_filled_binary.csv")
windows_df = pd.read_csv(ROOT / "data" / "stream" / "swat" / "windows_mixed.csv")

save_dir = ROOT / "data" / "stream" / "swat" / "window_binary_data"
save_dir.mkdir(parents=True, exist_ok=True)

lag = int(mining_config.lag)

rebuilt = []
missing_in_windows_csv = []

for win_id in window_ids:
    row = windows_df[windows_df["window_id"] == int(win_id)]
    if row.empty:
        missing_in_windows_csv.append(int(win_id))
        continue

    row = row.iloc[0]
    start_idx = int(row["start_idx"])
    end_idx = int(row["end_idx"])
    end_with_lag = min(len(X_binary), end_idx + lag)

    local_df = X_binary.iloc[start_idx:end_with_lag].reset_index(drop=True)
    out_path = save_dir / f"window_{int(win_id)}_binary.csv"
    local_df.to_csv(out_path, index=False)

    rebuilt.append({
        "window_id": int(win_id),
        "start_idx": start_idx,
        "end_idx": end_idx,
        "end_with_lag": end_with_lag,
        "binary_rows": len(local_df),
    })

rebuilt_df = pd.DataFrame(rebuilt)

print("lag =", lag)
print("missing_in_windows_csv:", missing_in_windows_csv)
print("num_rebuilt:", len(rebuilt_df))
display(rebuilt_df)


In [ ]:

# ===== 5. 辅助函数 =====
def confidence_to_weight(conf, eps=1e-6):
    conf = np.clip(float(conf), eps, 1 - eps)
    return float(np.log(conf / (1 - conf)))

def clipped_weight(conf, wmax=3.0):
    return float(np.clip(confidence_to_weight(conf), 0.0, wmax))

def relaxed_weight(score_relaxed, wmin=0.3, scale=30.0, wmax=3.0):
    w = scale * float(score_relaxed)
    return float(np.clip(w, wmin, wmax))

def decide_window_mode(attack_ratio: float) -> str:
    if attack_ratio < 0.30:
        return "balanced"
    elif attack_ratio >= 0.80:
        return "relaxed"
    else:
        return "standard"

def build_labeled_transactions(transactions, labels):
    assert len(transactions) == len(labels), f"transactions 与 labels 长度不一致: {len(transactions)} vs {len(labels)}"
    labeled_transactions = []
    for tx, y in zip(transactions, labels):
        label_token = LABEL_ATTACK if int(y) == 1 else LABEL_NORMAL
        labeled_transactions.append(list(tx) + [label_token])
    return labeled_transactions

def transactions_to_onehot(transactions):
    te = TransactionEncoder()
    arr = te.fit(transactions).transform(transactions)
    return pd.DataFrame(arr, columns=te.columns_)

def format_label_rules(freq_items, min_confidence=0.6):
    rules = association_rules(
        freq_items,
        metric="confidence",
        min_threshold=min_confidence
    ).copy()

    rules = rules[
        (rules["consequents"].apply(lambda x: len(x) == 1)) &
        (rules["consequents"].apply(lambda x: list(x)[0] in [LABEL_ATTACK, LABEL_NORMAL]))
    ].copy()

    if len(rules) == 0:
        return pd.DataFrame(columns=["antecedent_str", "consequent_str", "support", "confidence", "lift"])

    rules["antecedent_str"] = rules["antecedents"].apply(lambda x: " & ".join(sorted(list(x))))
    rules["consequent_str"] = rules["consequents"].apply(lambda x: list(x)[0])

    out = rules[["antecedent_str", "consequent_str", "support", "confidence", "lift"]].copy()
    out = out.sort_values(
        ["consequent_str", "confidence", "lift", "support"],
        ascending=[True, False, False, False]
    ).reset_index(drop=True)
    return out

def format_all_rules_for_fallback(freq_items):
    rules = association_rules(
        freq_items,
        metric="confidence",
        min_threshold=0.0
    ).copy()

    rules = rules[
        (rules["consequents"].apply(lambda x: len(x) == 1)) &
        (rules["consequents"].apply(lambda x: list(x)[0] in [LABEL_ATTACK, LABEL_NORMAL]))
    ].copy()

    if len(rules) == 0:
        return pd.DataFrame(columns=["antecedent_str", "consequent_str", "support", "confidence", "lift"])

    rules["antecedent_str"] = rules["antecedents"].apply(lambda x: " & ".join(sorted(list(x))))
    rules["consequent_str"] = rules["consequents"].apply(lambda x: list(x)[0])

    out = rules[["antecedent_str", "consequent_str", "support", "confidence", "lift"]].copy()
    out = out.sort_values(
        ["consequent_str", "confidence", "lift", "support"],
        ascending=[True, False, False, False]
    ).reset_index(drop=True)
    return out

def rank_relaxed_candidates(df, target_label, prior_prob, top_k):
    cand = df[df["consequent_str"] == target_label].copy()
    if len(cand) == 0:
        return cand

    cand["prior_prob"] = float(prior_prob)
    cand["conf_gain"] = cand["confidence"] - cand["prior_prob"]
    cand["score_relaxed"] = cand["conf_gain"] + 0.1 * (cand["lift"] - 1.0)

    cand = cand.sort_values(
        ["score_relaxed", "confidence", "lift", "support"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    return cand.head(top_k).copy()

def build_rule_pool_robust(label_rules, rules_all_formatted, attack_ratio, config):
    mode = decide_window_mode(attack_ratio)

    attack_rules = label_rules[label_rules["consequent_str"] == LABEL_ATTACK].reset_index(drop=True)
    normal_rules = label_rules[label_rules["consequent_str"] == LABEL_NORMAL].reset_index(drop=True)

    attack_top = attack_rules.head(config.top_attack_rules).copy()
    if len(attack_top) > 0:
        attack_top["formula"] = attack_top["antecedent_str"] + " => " + LABEL_ATTACK
        attack_top["weight"] = attack_top["confidence"].apply(lambda x: clipped_weight(x, wmax=config.attack_weight_cap))
        attack_top["target_label"] = 1

    if mode == "relaxed":
        normal_top = rank_relaxed_candidates(
            rules_all_formatted,
            target_label=LABEL_NORMAL,
            prior_prob=float(1.0 - attack_ratio),
            top_k=config.relaxed_normal_rules,
        ).copy()
        if len(normal_top) > 0:
            normal_top["consequent_str"] = LABEL_NORMAL
            normal_top["formula"] = normal_top["antecedent_str"] + " => " + LABEL_NORMAL
            normal_top["weight"] = normal_top["score_relaxed"].apply(
                lambda x: relaxed_weight(
                    x,
                    wmin=config.relaxed_normal_min_weight,
                    scale=config.relaxed_normal_weight_scale,
                    wmax=config.relaxed_normal_weight_cap,
                )
            )
            normal_top["target_label"] = 0
    else:
        normal_top = normal_rules.head(config.top_normal_rules).copy()
        if len(normal_top) > 0:
            normal_top["formula"] = normal_top["antecedent_str"] + " => " + LABEL_NORMAL
            normal_top["weight"] = normal_top["confidence"].apply(lambda x: clipped_weight(x, wmax=config.attack_weight_cap))
            normal_top["target_label"] = 0

    if mode == "balanced":
        need_attack = min(3, config.top_attack_rules)
        need_normal = min(3, config.top_normal_rules)
    elif mode == "relaxed":
        need_attack = max(1, min(config.top_attack_rules, 3))
        need_normal = max(1, config.relaxed_normal_rules)
    else:
        need_attack = max(1, min(config.top_attack_rules, 3))
        need_normal = max(1, min(config.top_normal_rules, 3))

    if len(attack_top) < need_attack:
        relaxed_attack_top = rank_relaxed_candidates(
            rules_all_formatted,
            target_label=LABEL_ATTACK,
            prior_prob=float(attack_ratio),
            top_k=max(need_attack, config.top_attack_rules if mode != "balanced" else need_attack),
        ).copy()
        if len(relaxed_attack_top) > 0:
            relaxed_attack_top["consequent_str"] = LABEL_ATTACK
            relaxed_attack_top["formula"] = relaxed_attack_top["antecedent_str"] + " => " + LABEL_ATTACK
            relaxed_attack_top["weight"] = relaxed_attack_top["score_relaxed"].apply(
                lambda x: relaxed_weight(
                    x,
                    wmin=config.relaxed_normal_min_weight,
                    scale=config.relaxed_normal_weight_scale,
                    wmax=config.attack_weight_cap,
                )
            )
            relaxed_attack_top["target_label"] = 1
            attack_top = relaxed_attack_top.copy()

    if len(normal_top) < need_normal:
        relaxed_normal_top = rank_relaxed_candidates(
            rules_all_formatted,
            target_label=LABEL_NORMAL,
            prior_prob=float(1.0 - attack_ratio),
            top_k=max(
                need_normal,
                config.relaxed_normal_rules if mode == "relaxed" else config.top_normal_rules
            ),
        ).copy()
        if len(relaxed_normal_top) > 0:
            relaxed_normal_top["consequent_str"] = LABEL_NORMAL
            relaxed_normal_top["formula"] = relaxed_normal_top["antecedent_str"] + " => " + LABEL_NORMAL
            relaxed_normal_top["weight"] = relaxed_normal_top["score_relaxed"].apply(
                lambda x: relaxed_weight(
                    x,
                    wmin=config.relaxed_normal_min_weight,
                    scale=config.relaxed_normal_weight_scale,
                    wmax=config.relaxed_normal_weight_cap,
                )
            )
            relaxed_normal_top["target_label"] = 0
            normal_top = relaxed_normal_top.copy()

    if len(attack_top) == 0 or len(normal_top) == 0:
        raise ValueError(f"规则池不完整: attack={len(attack_top)}, normal={len(normal_top)}, mode={mode}")

    mixed_rule_pool = pd.concat([
        attack_top[["antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"]],
        normal_top[["antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"]],
    ], axis=0, ignore_index=True)

    return mixed_rule_pool.reset_index(drop=True), mode

def build_ws_state(active_mask, weights, rule_scores, target_labels, rule_idx):
    num_rules = len(target_labels)
    active_weights = weights[active_mask > 0.5]
    if len(active_weights) == 0:
        mean_w = std_w = min_w = max_w = 0.0
    else:
        mean_w = float(active_weights.mean())
        std_w = float(active_weights.std())
        min_w = float(active_weights.min())
        max_w = float(active_weights.max())

    x = np.array([
        float(active_mask.mean()),
        mean_w,
        std_w,
        min_w,
        max_w,
        float(active_mask.sum()),
        float(num_rules),
        float(rule_idx / max(1, num_rules - 1)),
        float(active_mask[rule_idx]),
        float(weights[rule_idx]),
        float(rule_scores[rule_idx]),
        float(target_labels[rule_idx]),
    ], dtype=np.float32)
    return x

def get_logits(output):
    if isinstance(output, (tuple, list)):
        return output[0]
    if isinstance(output, dict):
        for k in ["policy_logits", "logits", "actor_logits"]:
            if k in output:
                return output[k]
    return output

def baseline_eval(state_dict, action_policy, random_trials=20, seed=42):
    num_rules = int(state_dict["num_rules"])
    target_labels = np.array(state_dict["target_labels"])

    def run_once(actions):
        env = SimpleRuleEnv(state_dict, step_size=0.5, max_steps=100)
        state = env.reset()
        total_reward = 0.0
        rows = []
        for rule_idx in range(num_rules):
            action = int(actions[rule_idx])
            next_state, reward, done, info = env.step(rule_idx, action)
            total_reward += float(reward)

            target_label = int(target_labels[rule_idx])
            is_correct = int((target_label == 1 and action == 0) or (target_label == 0 and action == 1))
            rows.append({
                "action": action,
                "target_label": target_label,
                "is_correct": is_correct,
            })
            state = next_state

        df = pd.DataFrame(rows)
        attack_mask = df["target_label"] == 1
        normal_mask = df["target_label"] == 0
        return {
            "attack_keep_rate": float((df.loc[attack_mask, "action"] == 0).mean()) if attack_mask.sum() > 0 else np.nan,
            "normal_disable_rate": float((df.loc[normal_mask, "action"] == 1).mean()) if normal_mask.sum() > 0 else np.nan,
            "selection_accuracy": float(df["is_correct"].mean()),
            "total_reward": float(total_reward),
        }

    if action_policy == "all_keep":
        actions = np.zeros(num_rules, dtype=int)
        return run_once(actions)

    if action_policy == "all_disable":
        actions = np.ones(num_rules, dtype=int)
        return run_once(actions)

    if action_policy == "random":
        rng = np.random.RandomState(seed)
        outs = []
        for _ in range(random_trials):
            actions = rng.randint(0, 2, size=num_rules)
            outs.append(run_once(actions))
        df = pd.DataFrame(outs)
        return {
            "attack_keep_rate": float(df["attack_keep_rate"].mean()),
            "normal_disable_rate": float(df["normal_disable_rate"].mean()),
            "selection_accuracy": float(df["selection_accuracy"].mean()),
            "total_reward_mean": float(df["total_reward"].mean()),
            "total_reward_std": float(df["total_reward"].std(ddof=0)),
        }


In [ ]:

# ===== 6. 单窗口正式实验 =====
def run_single_window_experiment_local(
    project_root: Path,
    window_id: int,
    mining_config: RuleMiningConfig,
    warmstart_config: WarmStartConfig,
    eval_config: EvalConfig,
):
    stream_dir = project_root / "data" / "stream" / "swat"
    logs_dir = project_root / "outputs" / "logs"
    models_dir = project_root / "outputs" / "models"

    for d in [logs_dir, models_dir]:
        d.mkdir(parents=True, exist_ok=True)

    binary_path = stream_dir / "window_binary_data" / f"window_{window_id}_binary.csv"
    if not binary_path.exists():
        raise FileNotFoundError(binary_path)

    df_binary = pd.read_csv(binary_path)

    windows_df_local = pd.read_csv(stream_dir / "windows_mixed.csv")
    row = windows_df_local[windows_df_local["window_id"] == int(window_id)]
    if row.empty:
        raise ValueError(f"windows_mixed.csv 中不存在 window_id={window_id}")
    row = row.iloc[0]

    start_idx = int(row["start_idx"])
    end_idx = int(row["end_idx"])

    y_all = pd.read_csv(project_root / "data" / "processed" / "swat" / "y_filled.csv").iloc[:, 0].astype(int).reset_index(drop=True)
    labels = y_all.iloc[start_idx:end_idx].reset_index(drop=True)

    transactions = build_transactions(df_binary, lag=mining_config.lag)
    if len(transactions) != len(labels):
        raise ValueError(f"transactions 与 labels 长度不一致: {len(transactions)} vs {len(labels)}")

    attack_ratio = float(labels.mean())

    labeled_transactions = build_labeled_transactions(transactions, labels.tolist())
    df_onehot = transactions_to_onehot(labeled_transactions)

    freq_items = fpgrowth(
        df_onehot,
        min_support=mining_config.min_support,
        use_colnames=True,
        max_len=2
    ).sort_values("support", ascending=False).reset_index(drop=True)

    label_rules = format_label_rules(freq_items, min_confidence=mining_config.min_confidence)
    rules_all_formatted = format_all_rules_for_fallback(freq_items)

    mixed_rule_pool, window_mode = build_rule_pool_robust(
        label_rules=label_rules,
        rules_all_formatted=rules_all_formatted,
        attack_ratio=attack_ratio,
        config=mining_config,
    )

    state_dict = build_initial_rule_state(mixed_rule_pool)

    active_mask = np.array(state_dict["active_mask"], dtype=np.float32)
    weights = np.array(state_dict["weights"], dtype=np.float32)
    rule_scores = np.array(state_dict["rule_scores"], dtype=np.float32)
    target_labels = np.array(state_dict["target_labels"], dtype=np.int64)
    num_rules = int(state_dict["num_rules"])

    X_ws = np.stack([
        build_ws_state(active_mask, weights, rule_scores, target_labels, i)
        for i in range(num_rules)
    ], axis=0)

    y_ws = np.array([0 if t == 1 else 1 for t in target_labels], dtype=np.int64)
    num_keep = int((y_ws == 0).sum())
    num_disable = int((y_ws == 1).sum())

    X_tensor = torch.tensor(X_ws, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_ws, dtype=torch.long).to(device)

    class_weights = torch.tensor(
        [1.0 / max(num_keep, 1), 1.0 / max(num_disable, 1)],
        dtype=torch.float32
    ).to(device)

    torch.manual_seed(warmstart_config.seed)
    model = ActorCriticNet(state_dim=12, action_dim=2, hidden_dim=64).to(device)
    optimizer = optim.Adam(model.parameters(), lr=warmstart_config.lr)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    for epoch in range(warmstart_config.epochs):
        model.train()
        optimizer.zero_grad()
        logits = get_logits(model(X_tensor))
        loss = criterion(logits, y_tensor)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        model.eval()
        logits = get_logits(model(X_tensor))
        pred = logits.argmax(dim=1)
        warmstart_train_accuracy = float((pred == y_tensor).float().mean().item())

    model_path = models_dir / f"window_{window_id}_actor_warmstart.pth"
    torch.save(model.state_dict(), model_path)

    env = SimpleRuleEnv(state_dict, step_size=0.5, max_steps=100)
    state = env.reset()

    records = []
    total_reward = 0.0

    for rule_idx in range(num_rules):
        x = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            logits = get_logits(model(x))
            action = int(torch.argmax(logits, dim=1).item())

        next_state, reward, done, info = env.step(rule_idx, action)
        total_reward += float(reward)

        target_label = int(target_labels[rule_idx])
        is_correct = int((target_label == 1 and action == 0) or (target_label == 0 and action == 1))

        records.append({
            "rule_idx": rule_idx,
            "action": action,
            "reward": float(reward),
            "target_label": target_label,
            "is_correct": is_correct,
            "weight": float(info["weight"]),
            "rule_score": float(info["rule_score"]),
            "active_after": int(info["active"]),
        })

        state = next_state

    trace_df = pd.DataFrame(records)
    trace_path = logs_dir / f"window_{window_id}_greedy_trace.csv"
    trace_df.to_csv(trace_path, index=False)

    attack_mask = trace_df["target_label"] == 1
    normal_mask = trace_df["target_label"] == 0

    attack_keep_rate = float((trace_df.loc[attack_mask, "action"] == 0).mean()) if attack_mask.sum() > 0 else np.nan
    normal_disable_rate = float((trace_df.loc[normal_mask, "action"] == 1).mean()) if normal_mask.sum() > 0 else np.nan
    selection_accuracy = float(trace_df["is_correct"].mean())

    baseline_keep = baseline_eval(state_dict, "all_keep", random_trials=eval_config.random_trials, seed=eval_config.seed)
    baseline_disable = baseline_eval(state_dict, "all_disable", random_trials=eval_config.random_trials, seed=eval_config.seed)
    baseline_random = baseline_eval(state_dict, "random", random_trials=eval_config.random_trials, seed=eval_config.seed)

    metrics_df = pd.DataFrame([{
        "num_rules": num_rules,
        "num_attack_rules": int(attack_mask.sum()),
        "num_normal_rules": int(normal_mask.sum()),
        "attack_keep_rate": attack_keep_rate,
        "normal_disable_rate": normal_disable_rate,
        "selection_accuracy": selection_accuracy,
        "greedy_total_reward": float(total_reward),
        "window_id": int(window_id),
        "attack_ratio": float(attack_ratio),
        "window_mode": window_mode,
        "warmstart_train_accuracy": warmstart_train_accuracy,
        "all_keep_reward": float(baseline_keep["total_reward"]),
        "all_disable_reward": float(baseline_disable["total_reward"]),
        "random_mean_reward": float(baseline_random["total_reward_mean"]),
        "random_std_reward": float(baseline_random["total_reward_std"]),
    }])

    metrics_path = logs_dir / f"window_{window_id}_metrics.csv"
    metrics_df.to_csv(metrics_path, index=False)

    return {
        "metrics_df": metrics_df,
        "trace_df": trace_df,
        "mixed_rule_pool": mixed_rule_pool,
        "state_dict": state_dict,
        "model_path": model_path,
        "metrics_path": metrics_path,
    }


In [ ]:

# ===== 7. 逐窗口运行，避免整批中断 =====
success_rows = []
fail_rows = []
success_ids = []

for win_id in window_ids:
    print("\n" + "=" * 80)
    print(f"running window {win_id} ...")

    try:
        result = run_single_window_experiment_local(
            project_root=ROOT,
            window_id=int(win_id),
            mining_config=mining_config,
            warmstart_config=warmstart_config,
            eval_config=eval_config,
        )

        row = result["metrics_df"].iloc[0].to_dict()
        success_rows.append(row)
        success_ids.append(int(win_id))

        print(
            f"[OK] window={win_id} "
            f"acc={row.get('selection_accuracy', np.nan):.6f} "
            f"attack_keep={row.get('attack_keep_rate', np.nan):.6f} "
            f"normal_disable={row.get('normal_disable_rate', np.nan):.6f}"
        )

    except Exception as e:
        fail_rows.append({
            "window_id": int(win_id),
            "error_type": type(e).__name__,
            "error_message": str(e),
            "traceback": traceback.format_exc(),
        })
        print(f"[FAIL] window={win_id} -> {type(e).__name__}: {e}")

print("\nsuccess_ids =", success_ids)
print("num_success =", len(success_rows))
print("num_fail =", len(fail_rows))


In [ ]:

# ===== 8. 保存 summary / fail log =====
summary_df = pd.DataFrame(success_rows)
fail_df = pd.DataFrame(fail_rows)

run_mode_local = globals().get("RUN_MODE", "smoke")

summary_name = (
    "formal_mixed_windows_summary_smoke.csv" if run_mode_local == "smoke"
    else ("formal_mixed_windows_summary_custom_batch.csv" if run_mode_local == "custom_batch" else "formal_mixed_windows_summary.csv")
)
fail_name = (
    "formal_mixed_windows_failures_smoke.csv" if run_mode_local == "smoke"
    else ("formal_mixed_windows_failures_custom_batch.csv" if run_mode_local == "custom_batch" else "formal_mixed_windows_failures.csv")
)
success_ids_name = (
    "formal_mixed_windows_success_ids_smoke.csv" if run_mode_local == "smoke"
    else ("formal_mixed_windows_success_ids_custom_batch.csv" if run_mode_local == "custom_batch" else "formal_mixed_windows_success_ids.csv")
)

summary_path = ROOT / "outputs" / "logs" / summary_name
fail_path = ROOT / "outputs" / "logs" / fail_name
success_ids_path = ROOT / "outputs" / "logs" / success_ids_name

summary_path.parent.mkdir(parents=True, exist_ok=True)

if len(summary_df) > 0:
    if "attack_ratio" in summary_df.columns:
        summary_df = summary_df.sort_values("attack_ratio").reset_index(drop=True)

    summary_df.to_csv(summary_path, index=False)
    print("saved summary:", summary_path)
    display(summary_df)

    if "selection_accuracy" in summary_df.columns:
        print("mean selection_accuracy =", summary_df["selection_accuracy"].mean())
    if "attack_keep_rate" in summary_df.columns:
        print("mean attack_keep_rate =", summary_df["attack_keep_rate"].mean())
    if "normal_disable_rate" in summary_df.columns:
        print("mean normal_disable_rate =", summary_df["normal_disable_rate"].mean())
else:
    print("没有成功窗口，summary_df 为空。")

if len(fail_df) > 0:
    fail_df.to_csv(fail_path, index=False)
    print("saved failures:", fail_path)
    display(fail_df[["window_id", "error_type", "error_message"]])
else:
    print("没有失败窗口。")

pd.DataFrame({"window_id": success_ids}).to_csv(success_ids_path, index=False)
print("saved success_ids:", success_ids_path)
print("run_mode_local =", run_mode_local)


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())

summary_path = ROOT / "outputs/logs/formal_mixed_windows_summary.csv"
save_clean_path = ROOT / "outputs/logs/formal_mixed_windows_summary_clean.csv"
save_group_path = ROOT / "outputs/logs/formal_mixed_windows_group_stats.csv"

df = pd.read_csv(summary_path).copy()

def attack_bin(x):
    if x < 0.30:
        return "low"
    elif x >= 0.80:
        return "high"
    else:
        return "mid"

df["attack_bin"] = df["attack_ratio"].apply(attack_bin)

clean_cols = [
    "window_id", "attack_ratio", "attack_bin", "window_mode",
    "num_rules", "num_attack_rules", "num_normal_rules",
    "attack_keep_rate", "normal_disable_rate", "selection_accuracy",
    "greedy_total_reward", "warmstart_train_accuracy",
    "all_keep_reward", "all_disable_reward",
    "random_mean_reward", "random_std_reward"
]
df_clean = df[clean_cols].sort_values("attack_ratio").reset_index(drop=True)
df_clean.to_csv(save_clean_path, index=False)

group_stats = df_clean.groupby("attack_bin")[[
    "attack_keep_rate", "normal_disable_rate",
    "selection_accuracy", "greedy_total_reward"
]].agg(["mean", "std", "count"])

group_stats.to_csv(save_group_path)

print("saved clean:", save_clean_path)
print("saved group:", save_group_path)
print("\n[df_clean]")
print(df_clean)
print("\n[group_stats]")
print(group_stats)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())

clean_path = ROOT / "outputs/logs/formal_mixed_windows_summary_clean.csv"
save_path = ROOT / "outputs/figures/formal_attack_ratio_vs_selection_accuracy.png"

df = pd.read_csv(clean_path).copy()
df = df.sort_values("attack_ratio").reset_index(drop=True)

marker_map = {"low": "o", "mid": "s", "high": "^"}

plt.figure(figsize=(7, 5))

for attack_bin in ["low", "mid", "high"]:
    part = df[df["attack_bin"] == attack_bin].copy()
    if len(part) == 0:
        continue

    plt.scatter(
        part["attack_ratio"],
        part["selection_accuracy"],
        s=70,
        marker=marker_map[attack_bin],
        label=f"{attack_bin} ({len(part)})"
    )

    for _, row in part.iterrows():
        plt.text(
            row["attack_ratio"] + 0.005,
            row["selection_accuracy"] + 0.002,
            str(int(row["window_id"])),
            fontsize=8
        )

plt.xlabel("Attack Ratio")
plt.ylabel("Selection Accuracy")
plt.title("Formal Experiment: Attack Ratio vs Selection Accuracy")
plt.ylim(0.74, 0.97)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("saved:", save_path)
print(df[["window_id", "attack_ratio", "attack_bin", "selection_accuracy", "normal_disable_rate"]])

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())

group_path = ROOT / "outputs/logs/formal_mixed_windows_group_stats.csv"
save_path = ROOT / "outputs/figures/formal_grouped_metrics_bar.png"

# 读取 group_stats.csv（两层表头）
group_df = pd.read_csv(group_path, header=[0, 1], index_col=0)

bins = list(group_df.index)

sel_mean = group_df[("selection_accuracy", "mean")].values
sel_std  = group_df[("selection_accuracy", "std")].fillna(0).values

nd_mean = group_df[("normal_disable_rate", "mean")].values
nd_std  = group_df[("normal_disable_rate", "std")].fillna(0).values

gr_mean = group_df[("greedy_total_reward", "mean")].values
gr_std  = group_df[("greedy_total_reward", "std")].fillna(0).values

x = np.arange(len(bins))
width = 0.24

plt.figure(figsize=(8, 5))

plt.bar(x - width, sel_mean, width, yerr=sel_std, capsize=4, label="Selection Accuracy")
plt.bar(x, nd_mean, width, yerr=nd_std, capsize=4, label="Normal Disable Rate")
plt.bar(x + width, gr_mean, width, yerr=gr_std, capsize=4, label="Greedy Total Reward")

plt.xticks(x, bins)
plt.xlabel("Attack Ratio Group")
plt.ylabel("Value")
plt.title("Formal Experiment: Group-wise Metrics")
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("saved:", save_path)
print(group_df)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
 
ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())

summary_path = ROOT / "outputs/logs/formal_mixed_windows_summary_clean.csv"
save_path = ROOT / "outputs/figures/formal_baseline_reward_comparison.png"

df = pd.read_csv(summary_path).copy()
df = df.sort_values("attack_ratio").reset_index(drop=True)

x = np.arange(len(df))
width = 0.25

plt.figure(figsize=(12, 5))

plt.bar(x - width, df["greedy_total_reward"], width, label="Greedy Reward")
plt.bar(x, df["random_mean_reward"], width, label="Random Mean Reward")
plt.bar(x + width, df["all_disable_reward"], width, label="All Disable Reward")

plt.xticks(x, df["window_id"].astype(str), rotation=45)
plt.xlabel("Window ID")
plt.ylabel("Reward")
plt.title("Formal Experiment: Reward Comparison with Baselines")
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("saved:", save_path)
print(df[[
    "window_id", "attack_ratio", "greedy_total_reward",
    "random_mean_reward", "all_disable_reward"
]])